# Day 014 — Exercise 3: Chunk by Sentences

**Goal:** Implement `chunk_by_sentences` — a chunker that always splits on sentence boundaries so retrieved chunks are coherent and citable.

**What you build:** A pure string function that groups whole sentences into chunks under `max_chars`, carrying `overlap_sentences` sentences into the next chunk.

In [ ]:
import re

## Your Task

Complete `chunk_by_sentences` below. The helper `split_sentences` is already provided — use it to break the text into a list of sentence strings before grouping them.

**Algorithm outline:**
1. Split `text` into sentences with `split_sentences`.
2. Walk through each sentence. Build a `candidate` string by joining `current` sentences plus the new one.
3. If `candidate` exceeds `max_chars` **and** `current` is non-empty, flush `current` as a finished chunk, then start a new `current` seeded with the last `overlap_sentences` sentences from the flushed chunk plus the new sentence.
4. Otherwise just append the sentence to `current`.
5. After the loop, flush whatever remains in `current`.

## Your Implementation

In [ ]:
def split_sentences(text: str) -> list[str]:
    """Split text into sentences on . ! ? boundaries."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]


def chunk_by_sentences(
    text: str,
    max_chars: int = 500,
    overlap_sentences: int = 1,
) -> list[str]:
    """
    Group whole sentences into chunks of at most max_chars characters.
    
    Args:
        text:               The document to chunk.
        max_chars:          Maximum characters per chunk.
        overlap_sentences:  Number of trailing sentences to carry into the next chunk.
    
    Returns:
        List of chunk strings; each chunk ends on a sentence boundary.
    """
    sentences = split_sentences(text)
    chunks = []
    current: list[str] = []

    # TODO: iterate sentences
    # TODO: if adding the next sentence would exceed max_chars (and current is non-empty),
    #       flush current as a chunk, then start fresh with the last overlap_sentences + new sentence
    # TODO: otherwise append to current
    # TODO: after the loop, flush whatever remains in current
    pass

In [ ]:
def _run_checks():
    total = 5
    passed = 0
    
    # Check 1: returns a list
    try:
        result = chunk_by_sentences("Hello world. This is a test.", max_chars=500)
        assert isinstance(result, list), f"Expected list, got {type(result)}"
        passed += 1
        print("\u2705 Check 1: returns a list")
    except Exception as e:
        print(f"\u274c Check 1: returns a list \u2014 {e}")
    
    # Check 2: text shorter than max_chars is a single chunk
    try:
        short = "Python is great. It is readable."
        result = chunk_by_sentences(short, max_chars=500)
        assert len(result) == 1, f"Short text should be 1 chunk, got {len(result)}"
        passed += 1
        print("\u2705 Check 2: text shorter than max_chars is a single chunk")
    except Exception as e:
        print(f"\u274c Check 2: single chunk \u2014 {e}")
    
    # Check 3: each chunk respects max_chars
    try:
        long_text = " ".join([f"Sentence number {i} is here." for i in range(30)])
        result = chunk_by_sentences(long_text, max_chars=100, overlap_sentences=0)
        for i, chunk in enumerate(result):
            assert len(chunk) <= 150, \
                f"Chunk {i} is {len(chunk)} chars, expected <= 150 (with some tolerance for long sentences)"
        passed += 1
        print("\u2705 Check 3: chunks respect max_chars")
    except Exception as e:
        print(f"\u274c Check 3: max_chars respected \u2014 {e}")
    
    # Check 4: multiple chunks produced for long text
    try:
        long_text = " ".join([f"Sentence number {i} ends here." for i in range(20)])
        result = chunk_by_sentences(long_text, max_chars=80, overlap_sentences=0)
        assert len(result) > 1, f"Long text should produce multiple chunks, got {len(result)}"
        passed += 1
        print("\u2705 Check 4: long text produces multiple chunks")
    except Exception as e:
        print(f"\u274c Check 4: multiple chunks \u2014 {e}")
    
    # Check 5: overlap_sentences carries last sentence into next chunk
    try:
        # 3 sentences, small max_chars forces splits; overlap=1 means last sentence of chunk N
        # should appear at start of chunk N+1
        sents = ["Alpha beta gamma.", "Delta epsilon zeta.", "Eta theta iota kappa."]
        text = " ".join(sents)
        result = chunk_by_sentences(text, max_chars=35, overlap_sentences=1)
        # We need at least 2 chunks for the overlap to be observable
        assert len(result) >= 2, f"Expected at least 2 chunks, got {len(result)}: {result}"
        # The last sentence of chunk 0 should appear in chunk 1 (overlap)
        last_sent_of_first = split_sentences(result[0])[-1]
        assert last_sent_of_first in result[1], \
            f"Overlap sentence {last_sent_of_first!r} not found in chunk 1: {result[1]!r}"
        passed += 1
        print("\u2705 Check 5: overlap_sentences carries last sentence into next chunk")
    except Exception as e:
        print(f"\u274c Check 5: overlap_sentences \u2014 {e}")
    
    if passed == total:
        print("🎉 Exercise complete!")
    print(f"\nScore: {passed}/{total}")

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def split_sentences(text: str) -> list[str]:
    """Split text into sentences on . ! ? boundaries."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]


def chunk_by_sentences(
    text: str,
    max_chars: int = 500,
    overlap_sentences: int = 1,
) -> list[str]:
    """Group whole sentences into chunks of at most max_chars characters."""
    sentences = split_sentences(text)
    chunks = []
    current: list[str] = []

    for sent in sentences:
        candidate = " ".join(current + [sent])
        if current and len(candidate) > max_chars:
            chunks.append(" ".join(current))
            current = (current[-overlap_sentences:] if overlap_sentences else []) + [sent]
        else:
            current.append(sent)

    if current:
        chunks.append(" ".join(current))
    return chunks
```

</details>